[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/27_vit_patch.ipynb)

# 🟠 Medium: ViT Patch Embedding

Реализуйте **patch embedding** для Vision Transformer. Transformer ожидает sequence токенов, поэтому изображение разбивается на неперекрывающиеся квадратные patches, каждый patch разворачивается в vector и проецируется в `embed_dim`.

### Из изображения в sequence
Для image `(B,C,H,W)` и patch size `P`, если `H` и `W` делятся на `P`, получается сетка `(H/P) × (W/P)` и

$$N = \frac{H}{P}\frac{W}{P}$$

patch tokens. Каждый patch содержит `C×P×P` значений. После Linear projection output имеет `(B,N,embed_dim)`, где порядок `N` должен последовательно обходить spatial grid.

### Reshape без перемешивания pixels
Нужно отделить высоту на `grid_h` и `P`, ширину на `grid_w` и `P`, затем переставить axes так, чтобы patch-grid шёл раньше внутренних coordinates patch. Только после этого dimensions `(C,P,P)` flatten в один feature vector. Один `reshape` без `permute` обычно смешивает pixels соседних patches.

### Linear и Conv2d — две эквивалентные точки зрения
Разворачивание patch плюс `Linear(C·P²,embed_dim)` эквивалентно `Conv2d` с `kernel_size=P` и `stride=P`, после которого spatial grid превращается в sequence. В упражнении реализуйте явно заданный Linear-путь, чтобы отработать shapes.

### Контракт
```python
class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim): ...
    def forward(self, x: Tensor) -> Tensor:
        # x: (B, C, H, W)
        # Returns: (B, num_patches, embed_dim)
```

### План реализации
1. Проверьте совместимость image и patch sizes.
2. Вычислите `num_patches = (img_size // patch_size) ** 2`.
3. Создайте Linear projection из `in_channels*P*P` в `embed_dim`.
4. В forward разделите spatial dimensions на grid и patch coordinates.
5. Переставьте axes, сформируйте `(B,N,C*P*P)` и примените projection.

### Быстрые самопроверки
- image `32×32`, patch `8×8` даёт `16` tokens;
- `num_patches` для `224/16` равно `196`;
- output shape `(B,N,embed_dim)` сохраняется для одного и нескольких channels;
- gradients проходят к input и projection parameters;
- patches не перекрываются и покрывают image полностью.

### Типичные ошибки
- `num_patches` считается как `img_size//patch_size`, без квадрата;
- отсутствует `permute` между reshape и flatten;
- flatten захватывает batch или patch-grid dimensions;
- Linear получает feature size `P²` без channel factor;
- output остаётся `(B,embed_dim,grid_h,grid_w)` вместо sequence.

### Официальные материалы и примеры
- [Torchvision ViT source](https://docs.pytorch.org/vision/main/_modules/torchvision/models/vision_transformer.html) — production patch projection через Conv2d и вычисление sequence length;
- [`torch.nn.Unfold`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Unfold.html) — извлечение image blocks в columns;
- [`torchvision.models.vit_b_16`](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.vit_b_16.html) — готовая ViT architecture с patch size 16.

### Вопросы для самопроверки
1. Почему patch projection превращает 2-D grid в 1-D sequence?
2. Зачем переставлять axes перед flatten?
3. Почему Conv2d с kernel=stride=P эквивалентна неперекрывающимся patches?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class PatchEmbedding(nn.Module):
    def __init__(self, img_size, patch_size, in_channels, embed_dim):
        super().__init__()
        pass  # self.num_patches, self.proj

    def forward(self, x):
        pass  # reshape to patches, project

In [ ]:
# 🧪 Debug
pe = PatchEmbedding(32, 8, 3, 64)
x = torch.randn(2, 3, 32, 32)
print('Output:', pe(x).shape)
print('Patches:', pe.num_patches)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('vit_patch')